# AI Job Impact ETL Pipeline

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AI Job Impact ETL") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark

## Read Data from HDFS

In [2]:
df = spark.read.csv(
    "hdfs://hadoop-namenode:9000/data/raw/ai_job_impact.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+-----------+---+------+---------------+-------------+-----------------+----------------+-----------------+---------------+-------------------+----------------+---------------+----------+-------------------+-----------+----------------+---------------------+
|Employee_ID|Age|Gender|Education_Level|     Industry|         Job_Role|Years_Experience|AI_Adoption_Level|Automation_Risk|Upskilling_Required|Salary_Before_AI|Salary_After_AI|Job_Status|Work_Hours_Per_Week|Remote_Work|Job_Satisfaction|Productivity_Change_%|
+-----------+---+------+---------------+-------------+-----------------+----------------+-----------------+---------------+-------------------+----------------+---------------+----------+-------------------+-----------+----------------+---------------------+
|      E0001| 50|Female|       Bachelor|    Marketing|  Content Creator|              26|             High|           High|                Yes|          106820|          95455|  Replaced|                 45|         No|    

قلل الضغط قبل الكتابة

In [4]:
df = df.repartition(2)

## Data Cleaning

In [5]:
df = df.dropDuplicates()

df = df.fillna({
    "Job_Satisfaction": 0,
    "Productivity_Change_%": 0
})

df.count()

2000

## Feature Engineering

In [6]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "Salary_Change",
    col("Salary_After_AI") - col("Salary_Before_AI")
)

df = df.withColumn(
    "Salary_Change_Percent",
    (col("Salary_Change") / col("Salary_Before_AI")) * 100
)

df = df.withColumn(
    "Age_Group",
    when(col("Age") < 30, "Young")
    .when((col("Age") >= 30) & (col("Age") < 50), "Mid")
    .otherwise("Senior")
)

df = df.withColumn(
    "Experience_Level",
    when(col("Years_Experience") < 3, "Junior")
    .when(col("Years_Experience") < 10, "Mid")
    .otherwise("Senior")
)

df.show(5)

+-----------+---+------+---------------+-------------+-----------------+----------------+-----------------+---------------+-------------------+----------------+---------------+----------+-------------------+-----------+----------------+---------------------+-------------+---------------------+---------+----------------+
|Employee_ID|Age|Gender|Education_Level|     Industry|         Job_Role|Years_Experience|AI_Adoption_Level|Automation_Risk|Upskilling_Required|Salary_Before_AI|Salary_After_AI|Job_Status|Work_Hours_Per_Week|Remote_Work|Job_Satisfaction|Productivity_Change_%|Salary_Change|Salary_Change_Percent|Age_Group|Experience_Level|
+-----------+---+------+---------------+-------------+-----------------+----------------+-----------------+---------------+-------------------+----------------+---------------+----------+-------------------+-----------+----------------+---------------------+-------------+---------------------+---------+----------------+
|      E1062| 54|Female|         M

## Aggregations

In [7]:
df.groupBy("Job_Status").count().show()

df.groupBy("Industry").avg("Salary_After_AI").show()

df.groupBy("Industry") \
  .avg("Productivity_Change_%") \
  .orderBy("avg(Productivity_Change_%)", ascending=False) \
  .show()

+----------+-----+
|Job_Status|count|
+----------+-----+
|  Replaced|  106|
| Unchanged| 1093|
|  Modified|  801|
+----------+-----+

+-------------+--------------------+
|     Industry|avg(Salary_After_AI)|
+-------------+--------------------+
|   Healthcare|   78702.02962962963|
|           IT|   79551.59731543624|
|      Finance|   76106.43333333333|
|    Marketing|   77597.72893772894|
|       Retail|     80376.963099631|
|Manufacturing|   78489.93898305084|
|    Education|   78322.73720136519|
+-------------+--------------------+

+-------------+--------------------------+
|     Industry|avg(Productivity_Change_%)|
+-------------+--------------------------+
|Manufacturing|        10.697389830508468|
|    Marketing|        10.332527472527474|
|    Education|         10.01450511945392|
|      Finance|         9.565766666666672|
|       Retail|          9.51287822878229|
|   Healthcare|         9.307259259259258|
|           IT|         9.061073825503351|
+-------------+-------------

## Star Schema

In [8]:
dim_employee = df.select(
    "Employee_ID",
    "Age",
    "Gender",
    "Education_Level"
).dropDuplicates()

dim_job = df.select(
    "Job_Role",
    "Industry",
    "Years_Experience"
).dropDuplicates()

dim_ai = df.select(
    "AI_Adoption_Level",
    "Automation_Risk",
    "Upskilling_Required"
).dropDuplicates()

fact_table = df.select(
    "Employee_ID",
    "Job_Role",
    "AI_Adoption_Level",
    "Job_Status",
    "Salary_Before_AI",
    "Salary_After_AI",
    "Salary_Change",
    "Salary_Change_Percent",
    "Productivity_Change_%",
    "Work_Hours_Per_Week"
)

## Save نتائج

In [9]:
dim_employee.coalesce(1).write.mode("overwrite").parquet("data/dim_employee")
dim_job.coalesce(1).write.mode("overwrite").parquet("data/dim_job")
dim_ai.coalesce(1).write.mode("overwrite").parquet("data/dim_ai")
fact_table.coalesce(1).write.mode("overwrite").parquet("data/fact_table")

In [10]:
fact_table.groupBy("Job_Status").count().show()

+----------+-----+
|Job_Status|count|
+----------+-----+
|  Replaced|  106|
| Unchanged| 1093|
|  Modified|  801|
+----------+-----+

